# 07 — PyTorch `nn.Module`: The Clean Way to Build Networks

## Same Network, Cleaner Code

In notebook 04 we managed W1, b1, W2, b2 as separate variables and wrote  
every gradient update by hand.

PyTorch's `nn.Module` system lets us:
- **Declare the architecture** in a class
- **Let PyTorch track parameters** (no more W1, b1, ...)
- **Use `optimizer.step()`** instead of manual weight updates

We'll solve XOR again — same problem, maybe 3× less code.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

# XOR dataset
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

## Defining the Network

We subclass `nn.Module` and define:
- **`__init__`** — create the layers
- **`forward`** — describe how data flows through them

PyTorch automatically discovers all parameters (weights + biases)  
inside `nn.Linear` layers.

`nn.Linear(in, out)` creates a fully-connected layer:  
it holds a weight matrix of shape `(in, out)` and a bias vector of size `out`.

In [ ]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 2 inputs → 3 hidden neurons
        self.hidden = nn.Linear(2, 3)
        # 3 hidden neurons → 1 output
        self.output = nn.Linear(3, 1)

    def forward(self, x):
        x = torch.sigmoid(self.hidden(x))   # hidden layer + activation
        x = torch.sigmoid(self.output(x))   # output layer + activation
        return x

model = XORNet()
print('Network architecture:')
print(model)
print()
total_params = sum(p.numel() for p in model.parameters())
print(f'Total trainable parameters: {total_params}')
print('  (W1: 2×3=6, b1: 3, W2: 3×1=3, b2: 1  → 6+3+3+1 = 13)')

## The Training Loop: Three Standard Steps

Every PyTorch training loop follows the same pattern:

```python
# 1. Forward pass — compute predictions
y_pred = model(X)
loss   = criterion(y_pred, y)

# 2. Backward pass — compute gradients
optimizer.zero_grad()   # clear old gradients first!
loss.backward()         # compute new gradients

# 3. Update — nudge every parameter in the right direction
optimizer.step()
```

We use **SGD** (Stochastic Gradient Descent) with `lr=0.5`, same as before.  
The **MSELoss** criterion computes Mean Squared Error for us.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

losses = []

for step in range(5000):
    # 1. Forward
    y_pred = model(X)
    loss   = criterion(y_pred, y)

    # 2. Backward
    optimizer.zero_grad()   # always clear before backward!
    loss.backward()

    # 3. Update
    optimizer.step()

    losses.append(loss.item())

print(f'Final loss: {losses[-1]:.6f}')

In [ ]:
print('XOR predictions after training:')
with torch.no_grad():
    preds = model(X)

for inputs, pred, expected in zip(X, preds, y):
    a, b = int(inputs[0]), int(inputs[1])
    print(f'  {a} XOR {b}  → {pred.item():.3f}  (expected {int(expected.item())})')

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(losses, color="mediumseagreen")
plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.title("XOR with nn.Module — loss curve")
plt.tight_layout()
plt.show()

## Decision Boundary

In [ ]:
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200),
                     np.linspace(-0.3, 1.3, 200))
grid_np = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
grid_t  = torch.tensor(grid_np)

with torch.no_grad():
    probs = model(grid_t).numpy().reshape(xx.shape)

plt.figure(figsize=(5, 5))
plt.contourf(xx, yy, probs, levels=50, cmap="RdYlGn", alpha=0.7)
plt.colorbar(label="P(XOR = 1)")

y_flat = y.numpy().flatten()
X_np   = X.numpy()
for (a, b_pt), label in zip(X_np, y_flat):
    color = "green" if label == 1 else "red"
    marker = "o" if label == 1 else "s"
    plt.scatter(a, b_pt, c=color, marker=marker, s=150,
               edgecolors="black", linewidths=1.5, zorder=5)

plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xticks([0, 1]); plt.yticks([0, 1])
plt.xlabel("Input A"); plt.ylabel("Input B")
plt.title("nn.Module XOR — decision boundary")
plt.tight_layout()
plt.show()

## Key Lessons

1. **`nn.Module`** packages weights into a reusable, inspectable object.
2. **`nn.Linear`** handles weights and biases for a fully-connected layer.
3. The **three-step loop** (forward → zero_grad → backward → step)  
   is the universal PyTorch training pattern.
4. Real models (ResNets, Transformers) use the exact same loop —  
   just with much larger architectures.